# M5 Demand Forecasting + Inventory Optimization - Colab runner
Run the cells in order. **Runtime -> Change runtime type -> CPU is fine** (LightGBM runs on CPU here).
Step 0: upload `m5_forecasting.zip` (the project) when asked.

In [ ]:
# 1. Upload the project zip and install
from google.colab import files
up = files.upload()            # choose m5_forecasting.zip
!unzip -q -o m5_forecasting.zip
%cd m5_forecasting
!pip install -q -r requirements.txt -r requirements-app.txt

## 2a. Quick test on synthetic data (no Kaggle needed, ~4 min)

In [ ]:
!python -m src.run --synthetic

## 2b. Real M5 data
First accept the rules at kaggle.com/competitions/m5-forecasting-accuracy and create an API token (Kaggle -> Settings -> API).
Do NOT print the token or share this notebook with the key visible.

In [ ]:
import os
from google.colab import files
k = files.upload()             # choose kaggle.json  (do not print it!)
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!mkdir -p data/raw
!kaggle competitions download -c m5-forecasting-accuracy -p data/raw
!unzip -q -o data/raw/m5-forecasting-accuracy.zip -d data/raw
!ls data/raw

In [ ]:
# Optional: change sample size (free Colab RAM ~12GB handles ~1000-5000 series)
# !sed -i 's/n_series: 1000/n_series: 2000/' config.yaml
!python -m src.run --config config.yaml

## 3. Summarise and view results

In [ ]:
!python -m src.summarize > /dev/null
from IPython.display import Image, display, Markdown
display(Markdown(open('docs/RESULTS.md').read().split('![forecasts]')[0]))
for f in ['forecast_examples','cost_vs_service','sensitivity','shap_importance']:
    display(Image(f'docs/{f}.png'))

## 4. Streamlit app (via a free localtunnel link)
Run the next two cells. Open the printed link; if it asks for a password, paste the IP shown.

In [ ]:
import urllib.request
print("Tunnel password (public IP):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip())
!npm install -q localtunnel
!streamlit run app/streamlit_app.py &>/content/streamlit.log &

In [ ]:
!npx localtunnel --port 8501

## 5. Download everything (results + docs) to push to GitHub

In [ ]:
!zip -qr outputs.zip results docs
files.download('outputs.zip')